# 01 – Charging stations: JSON → clean CSV
Input: `data/raw/ontario_ev_stations.json` (NLR / NRCan Alternative Fuel Stations API, Ontario, electric)

Output: `data/processed/ontario_ev_stations_clean.csv`

In [ ]:
# In Google Colab: upload ontario_ev_stations.json when prompted
from google.colab import files
uploaded = files.upload()

In [ ]:
import json
import pandas as pd

with open("ontario_ev_stations.json", encoding="utf-8") as f:
    data = json.load(f)
print("Total results reported by API:", data["total_results"])
stations = data["fuel_stations"]
print("Stations in file:", len(stations))

df = pd.json_normalize(stations)
keep = {
    "id": "station_id", "station_name": "station_name",
    "street_address": "street_address", "city": "city",
    "state": "province", "zip": "postal_code",
    "latitude": "latitude", "longitude": "longitude",
    "ev_network": "network", "ev_connector_types": "connector_types",
    "ev_level1_evse_num": "level1_ports", "ev_level2_evse_num": "level2_ports",
    "ev_dc_fast_num": "dc_fast_ports", "access_code": "access",
    "status_code": "status", "facility_type": "facility_type",
    "open_date": "open_date", "date_last_confirmed": "date_last_confirmed",
}
df = df[list(keep)].rename(columns=keep)

## Cleaning steps
1. Connector list → comma-separated text
2. Null port counts → 0 (not reported = no ports of that level)
3. Total ports = Level 1 + Level 2 + DC fast
4. Standardize city names and postal codes; derive FSA (first 3 characters)
5. Parse open date; drop duplicate station IDs

In [ ]:
df["connector_types"] = df["connector_types"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else x)
for c in ["level1_ports", "level2_ports", "dc_fast_ports"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
df["total_ports"] = df["level1_ports"] + df["level2_ports"] + df["dc_fast_ports"]
df["city"] = df["city"].str.strip().str.title()
df["postal_code"] = df["postal_code"].str.upper().str.replace(" ", "", regex=False)
df["fsa"] = df["postal_code"].str[:3]
df["open_date"] = pd.to_datetime(df["open_date"], errors="coerce")
before = len(df)
df = df.drop_duplicates(subset="station_id")
print("Duplicates removed:", before - len(df))
print("Clean stations:", len(df), "| Total ports:", df["total_ports"].sum())
df.head()

In [ ]:
df.to_csv("ontario_ev_stations_clean.csv", index=False)
files.download("ontario_ev_stations_clean.csv")